# 🤖 **Introducción a la Regresión Logística**

La **regresión logística** es un modelo de aprendizaje supervisado utilizado para problemas de **clasificación binaria**, donde la variable de salida $y$ solo puede tomar dos valores, típicamente $0$ o $1$.



A diferencia de la regresión lineal, que modela la salida como una combinación lineal de las variables predictoras, la regresión logística utiliza una función **sigmoide** para mapear cualquier valor real a un intervalo entre $0$ y $1$, interpretado como una **probabilidad**.

### 📘 **Modelo del clasificador**

Sea $\mathbf{x} = (x_1, x_2, \ldots, x_n)$ un vector de entrada (variables predictoras) y $\boldsymbol{\beta} = (\beta_0, \beta_1, \ldots, \beta_n)$ el vector de parámetros. El modelo predice la probabilidad de que $y = 1$ como:

$$
\hat{y}(y = 1| x) = \sigma(z) = \frac{1}{1 + e^{-z}}, \quad \text{donde } z = \beta_0 + \beta_1 x_1 + \cdots + \beta_n x_n = \boldsymbol{\beta}^T \mathbf{x}
$$

In [ ]:
import numpy as np

#### 👇🏻 **Evaluación de la función sigmoide**

In [ ]:
# Datos (x, y)
x = np.array([0, 1, 2])
y = np.array([0, 0, 1])

In [ ]:
# Elección de parámetros
beta_0 = 5
beta_1 = 1

In [ ]:
# Definición combinación lineal
def linear_comb(beta_0, beta_1, x):
  return beta_0 + beta_1 * x

In [ ]:
# Función sigmoide
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
# Evaluación
y_hat = sigmoid(linear_comb(beta_0, beta_1, x))
y_hat

#### 🧮 **Función de costo o *log-loss***

Para ajustar los parámetros $\boldsymbol{\beta}$, se minimiza la siguiente función de costo:

$$
J(\boldsymbol{\beta}) = -\frac{1}{m} \sum_{i=1}^m \left( y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \right)
$$

donde $m$ es el número de ejemplos de entrenamiento y $\hat{y}_i = \sigma(\boldsymbol{\beta}^T \mathbf{x}_i)$.

In [ ]:
# Elección de parámetros
beta_0 = 7
beta_1 = 2

z = linear_comb(beta_0, beta_1, x)

In [ ]:
# Función de costo (log-loss)
def J(beta_0, beta_1, x, y):
  m = len(x)
  z = linear_comb(beta_0, beta_1, x)
  y_hat = sigmoid(z)

  cost = -(1/m) * np.sum(y * np.log(y_hat + 1e-9) + (1 - y) * np.log(1 - y_hat + 1e-9))

  return cost

In [ ]:
J(beta_0, beta_1, x, y)

#### 🤏🏻 **Minimización**

Para el caso de un solo predictor (una sola variable $x$), tenemos dos parámetros. Las derivadas parciales de $J(\beta_0, \beta_1)$ con respecto a cada uno son:

- Derivada con respecto a $\beta_0$:

$$
\frac{\partial J}{\partial \beta_0} = \frac{1}{m} \sum_{i=1}^m (\hat{y}_i - y_i)
$$

- Derivada con respecto a $\beta_1$:

$$
\frac{\partial J}{\partial \beta_1} = \frac{1}{m} \sum_{i=1}^m (\hat{y}_i - y_i) \cdot x_i
$$

Estas expresiones definen la dirección en que debemos ajustar $\beta_0$ y $\beta_1$ para reducir el valor de la función de costo en cada iteración del algoritmo.

A continuación, implementaremos estas derivadas en código para realizar una simulación con pocos datos.

Una vez definida la función de costo para la regresión logística podemos comenzar el proceso de optimización para encontrar los parámetros $\beta_0$ y $\beta_1$ que **minimicen esta función**.

Este proceso se realiza mediante el algoritmo de **gradiente descendente**, que ajusta los valores de los parámetros en la dirección opuesta al gradiente de la función de costo.

In [ ]:
# Elección de parámetros
beta_0 = 5
beta_1 = 1

In [ ]:
# Número de iteraciones y tasa de aprendizaje
epochs = 1000
alpha = 0.1
m = len(x)

for epoch in range(epochs):
    z = beta_0 + beta_1 * x
    y_hat = sigmoid(z)

    # Función de costo (log-loss)
    cost = J(beta_0, beta_1, x, y)

    # Gradientes
    d_beta_0 = (1/m) * np.sum(y_hat - y)
    d_beta_1 = (1/m) * np.sum((y_hat - y) * x)

    # Actualización de parámetros
    beta_0 -= alpha * d_beta_0
    beta_1 -= alpha * d_beta_1

    # Mostrar cada 100 iteraciones
    if epoch % 100 == 0:
        print(f"Iteración {epoch}: Costo = {cost:.4f}, β₀ = {beta_0:.4f}, β₁ = {beta_1:.4f}")

In [ ]:
beta_0, beta_1


### ✅ **Regla de decisión**


Una vez entrenado el modelo, se clasifica un nuevo ejemplo aplicando la siguiente regla:

$$
\hat{y}_{\text{clasificada}} =
\begin{cases}
1 & \text{si } \hat{y} \geq 0.5 \\
0 & \text{si } \hat{y} < 0.5
\end{cases}
$$

Este umbral puede ajustarse dependiendo del problema.


In [ ]:
# Elección de parámetros
beta_0 = 0.0
beta_1 = 0.0

In [ ]:
# Predicción final
def predecir(x_nuevo):
  return sigmoid(linear_comb(beta_0, beta_1, x_nuevo))

In [ ]:
x = [-2, -1, 0, 1, 2, 3]

In [ ]:
print("Predicciones:\n")
for xi in x:
    prob = predecir(xi)
    pred = 1 if prob >= 0.5 else 0
    print(f"x = {xi}, prob = {prob:.4f}, clase predicha = {pred}")

### ⚙️ **Implementación con `scikit-learn`**



La linrería `scikit-learn` permite entrenar un modelo de regresión logística con muy pocas líneas de código, ocultando la complejidad del entrenamiento interno (optimización del costo, cálculo de gradientes, etc.).

Una vez entrenado el modelo, podemos obtener:

- La **probabilidad predicha** de que $y = 1$ con `.predict_proba()`
- La **clase final** con `.predict()`, que aplica automáticamente la regla vista anteriormente.

Esta regla puede modificarse si se desea ajustar el umbral de clasificación manualmente.

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
# Datos simples
X = np.array([[0, 6, 7], [1, 7, 9], [2, 5, 10]])  # scikit-learn espera X en forma de matriz
y = np.array([0, 0, 1])

In [ ]:
# Crear y entrenar el modelo
modelo = LogisticRegression() # LBGFS (quasi-Newton) Limited-memory Broyden–Fletcher–Goldfarb–Shanno
modelo.fit(X, y)

In [ ]:
modelo.coef_, modelo.intercept_

In [ ]:
modelo.predict_proba([[0, 2, 9]])[0][1]

In [ ]:
modelo.predict([[0, 2, 9]])[0]

In [ ]:
# Predicciones
print("Predicciones:")
for xi in X:
    prob = modelo.predict_proba([xi])[0][1]  # probabilidad de clase 1
    pred = modelo.predict([xi])[0]
    print(f"x = {xi[0]}, prob = {prob:.4f}, clase predicha = {pred}")

In [ ]:
from sklearn.linear_model import SGDClassifier

In [ ]:
modelo1 = SGDClassifier(loss = 'log_loss', random_state = 42)
modelo1.fit(X, y)

In [ ]:
# Predicciones
print("Predicciones:")
for xi in X:
    prob = modelo1.predict_proba([xi])[0][1]  # probabilidad de clase 1
    pred = modelo1.predict([xi])[0]
    print(f"x = {xi[0]}, prob = {prob:.4f}, clase predicha = {pred}")

In [ ]:
modelo1.coef_, modelo1.intercept_

### 🌸 **Ejemplo con dataset real: Iris**


Usamos el dataset **Iris**, uno de los más conocidos en Machine Learning, para aplicar regresión logística. Este dataset tiene 3 clases de flores, pero aquí trabajamos con un **problema binario**: distinguir entre *Setosa* y *Versicolor*.

El **dataset Iris** contiene 150 observaciones de flores de la especie *iris*, divididas en tres clases:

- **Setosa** (0)
- **Versicolor** (1)
- **Virginica** (2)

Cada flor está representada por **4 características numéricas**:

| Variable | Descripción |
|----------|-------------|
| sepal length (cm) | Largo del sépalo |
| sepal width (cm)  | Ancho del sépalo |
| petal length (cm) | Largo del pétalo |
| petal width (cm)  | Ancho del pétalo |


**Proceso**:
- Se seleccionan dos clases: 0 (Setosa) y 1 (Versicolor)
- Se escogen dos características: largo y ancho del sépalo
- Se estandarizan los datos
- Se entrena un modelo de regresión logística
- Se evalúa el desempeño en un conjunto de prueba

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
iris = load_iris(as_frame = True)

iris.frame['target'].value_counts()

In [ ]:
# Cargar el dataset
iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names

In [ ]:
# Para hacerlo binario: clasificamos "setosa" (0) vs "no setosa" (1 o 2)
# Solo nos quedamos con las clases 0 y 1
X = X[y != 2]
y = y[y != 2]

In [ ]:
# Seleccionamos solo dos características para visualizar (sepal length y sepal width)
X = X[:, :2]
X.shape

**Escalamineto min-max**

El escalamiento min-max (o escalamiento min-max) es una técnica de preprocesamiento de datos que transforma los valores numéricos para que queden acotados dentro de un rango específico, generalmente entre 0 y 1

$$x' = \frac{x-x_{min}}{x_{max} - x_{min}}$$

In [ ]:
# Dividir datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, test_size=0.20)

# Estandarizar los datos
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.fit_transform(X_test)

In [ ]:
# Entrenar el modelo
modelo = LogisticRegression()
modelo.fit(X_train_std, y_train)

# Imprimir exactitud
print(f"Precisión en test: {modelo.score(X_test_std, y_test):.2f}")

In [ ]:
modelo.predict(X_test_std) == y_test

#### 〽️ **Visualización de la frontera de decisión**

In [ ]:
# Visualización de la frontera de decisión
x_min, x_max = X_train_std[:, 0].min() - 1, X_train_std[:, 0].max() + 1
y_min, y_max = X_train_std[:, 1].min() - 1, X_train_std[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

In [ ]:
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3)
plt.scatter(X_train_std[:, 0], X_train_std[:, 1], c=y_train, edgecolors='k')
plt.title("Frontera de decisión de regresión logística")
plt.xlabel("Largo del sépalo (estandarizado)")
plt.ylabel("Ancho del sépalo (estandarizado)")
plt.grid(True)
plt.show()

### 🧶 **Clasificación NO Binaria**

Hasta ahora, hemos trabajado con regresión logística en su forma **binaria**, es decir, clasificando entre **dos posibles clases** (por ejemplo, *Setosa* vs *No Setosa*).

Sin embargo, muchos problemas del mundo real implican **más de dos clases posibles**. Por ejemplo, en el dataset *Iris*, queremos distinguir entre **tres tipos de flores**:

- **Setosa** (clase 0)
- **Versicolor** (clase 1)
- **Virginica** (clase 2)

🧠 **¿Puede la regresión logística resolver problemas multiclase?**

Sí. Aunque la versión original de regresión logística fue diseñada para clasificación binaria, puede extenderse a problemas de múltiples clases usando dos estrategias:

- **One-vs-Rest (OvR)**: entrena un clasificador binario por cada clase contra las demás.
- **Multinomial (Softmax)**: entrena un solo modelo que calcula una **probabilidad para cada clase** usando la función **softmax**.

---

🔢 La **función que calcula las probabilidades** en el caso multiclase es la **función softmax**:

$$
P(y = k \mid \mathbf{x}) = \frac{e^{\boldsymbol{\beta}_k^T \mathbf{x}}}{\sum_{j=1}^K e^{\boldsymbol{\beta}_j^T \mathbf{x}}}
$$

🧮 La **función de costo** que se minimiza durante el entrenamiento se llama **log-loss multiclase** o **entropía cruzada**:

$$
J(\boldsymbol{\beta}) = -\frac{1}{m} \sum_{i=1}^m \sum_{k=1}^K y_{ik} \ln\left( \hat{y}_{ik} \right)
$$


⚙️ **¿Qué hace `scikit-learn`?**

Cuando usamos `LogisticRegression` con:

```python
LogisticRegression(multi_class='multinomial', solver='lbfgs')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cargar el dataset completo
iris = load_iris()
X = iris.data#[:, :2]  # Usamos las dos primeras variables para visualizar en 2D
y = iris.target       # Las tres clases: 0, 1, 2

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, train_size = 0.7)

# Estandarizar
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

# Modelo multinomial (!!!!)
modelo = LogisticRegression(multi_class='multinomial', solver='lbfgs')
modelo.fit(X_train_std, y_train)

# Precisión
print(f"Precisión en test: {modelo.score(X_test_std, y_test):.2f}")

In [ ]:
# Crear malla para graficar fronteras de decisión (con las 2 primeras variables)
x_min, x_max = X_train_std[:, 0].min() - 1, X_train_std[:, 0].max() + 1
y_min, y_max = X_train_std[:, 1].min() - 1, X_train_std[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300)
)

# Rellenar las características 3 y 4 con ceros (su valor medio estandarizado)
zeros = np.zeros(xx.ravel().shape)
grid_4d = np.c_[xx.ravel(), yy.ravel(), zeros, zeros]

# Predecir sobre la matriz de 4 características
Z = modelo.predict(grid_4d)
Z = Z.reshape(xx.shape)

# Graficar
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
scatter = plt.scatter(
    X_train_std[:, 0],
    X_train_std[:, 1],
    c=y_train,
    edgecolors="k",
    cmap=plt.cm.coolwarm,
)
plt.xlabel("Largo del sépalo (estandarizado)")
plt.ylabel("Ancho del sépalo (estandarizado)")
plt.title("Frontera de decisión - Regresión Logística Multiclase")
plt.grid(True)
plt.legend(*scatter.legend_elements(), title="Clases")
plt.show()